#### Cancer-specific hyperparams

In [6]:
import os
import json
import pandas as pd
import numpy as np

In [7]:
result_dir = "experiment/cancer_specific/transformer"

In [8]:
record = [d.name for d in os.scandir(result_dir) if d.is_dir()]
## change the dates here
record = [r for r in record if r.startswith("2024-08-29") or r.startswith("2024-08-30")]
record = sorted(record, key=str.lower)
print(record)

rank_res = []
for r in record:
    result_all_cancer = []
    for cancer in ["BRCA", "CESC", "COAD", "KIRC", "LAML", "LUAD", "SKCM"]:
        result = pd.read_csv(os.path.join(result_dir, r, f"result/train_result_{cancer}.csv"))
        avg_aupr = float(result["test_aupr"].iloc[-1].split(' ')[0])
        result_all_cancer.append(avg_aupr)
    avg_avg_aupr = np.mean(result_all_cancer)
    with open(os.path.join(result_dir, r, "params.json"), 'r') as f:
        params = json.load(f)
    rank_res.append({
        'name': r,
        'aupr': avg_avg_aupr,
        ## hyperparams
        'metrics': "transformer_lr="+str(params['transformer_lr'])+";predictor_lr="+str(params['predictor_lr'])+";n="+str(params['n'])+";num_layers="+str(params['num_layers']),
    })

['2024-08-29-12:29:36', '2024-08-30-10:22:56', '2024-08-30-10:59:11', '2024-08-30-11:35:46', '2024-08-30-12:17:31', '2024-08-30-12:56:53', '2024-08-30-13:35:59', '2024-08-30-14:14:48', '2024-08-30-15:10:33', '2024-08-30-16:09:29', '2024-08-30-17:01:14', '2024-08-30-17:59:12', '2024-08-30-18:58:16', '2024-08-30-20:11:01', '2024-08-30-22:09:47']


In [9]:
rank_res_cancer_specific = pd.DataFrame(rank_res)
rank_res_cancer_specific = rank_res_cancer_specific.sort_values(by=["aupr"],ascending=False).reset_index()
rank_res_cancer_specific.head()

,index,name,aupr,metrics
0,1,2024-08-30-10:22:56,0.359543,transformer_lr=0.0005;predictor_lr=0.001;n=10;...
1,7,2024-08-30-14:14:48,0.359043,transformer_lr=0.0005;predictor_lr=0.001;n=20;...
2,2,2024-08-30-10:59:11,0.354871,transformer_lr=0.0005;predictor_lr=0.0005;n=10...
3,8,2024-08-30-15:10:33,0.354671,transformer_lr=0.0005;predictor_lr=0.0005;n=20...
4,5,2024-08-30-12:56:53,0.352871,transformer_lr=5e-05;predictor_lr=0.001;n=10;n...


#### Mixed-cancer hyperparams

In [10]:
result_dir = "experiment/mix/transformer"

record = [d.name for d in os.scandir(result_dir) if d.is_dir()]
record = [r for r in record if r.startswith("2024-08-30") or r.startswith("2024-08-31")]
record = sorted(record, key=str.lower)
print(record)

rank_res = []
for r in record:
    result = pd.read_csv(os.path.join(result_dir, r, f"result/train_result_all.csv"))
    avg_aupr = float(result["test_aupr"].iloc[-1].split(' ')[0])
    with open(os.path.join(result_dir, r, "params.json"), 'r') as f:
        params = json.load(f)
    rank_res.append({
        'name': r,
        'aupr': avg_aupr,
        'metrics': "transformer_lr="+str(params['transformer_lr'])+";predictor_lr="+str(params['predictor_lr'])+";n="+str(params['n'])+";num_layers="+str(params['num_layers']),
    })

['2024-08-30-07:47:51', '2024-08-30-10:23:14', '2024-08-30-10:53:02', '2024-08-30-11:21:22', '2024-08-30-11:51:36', '2024-08-30-12:21:37', '2024-08-30-12:53:29', '2024-08-30-13:29:45', '2024-08-30-14:21:05', '2024-08-30-15:11:53', '2024-08-30-16:04:01', '2024-08-30-16:58:18', '2024-08-30-17:53:10', '2024-08-30-18:49:36', '2024-08-30-20:36:50', '2024-08-30-22:22:15', '2024-08-31-00:09:10', '2024-08-31-02:10:31', '2024-08-31-04:10:28', '2024-08-31-06:19:30', '2024-08-31-06:51:43', '2024-08-31-07:25:55', '2024-08-31-07:59:42', '2024-08-31-08:36:11', '2024-08-31-09:12:22', '2024-08-31-09:48:48', '2024-08-31-10:45:11', '2024-08-31-11:44:01', '2024-08-31-12:37:22', '2024-08-31-13:36:09', '2024-08-31-14:34:45']


In [11]:
rank_res_mix = pd.DataFrame(rank_res)
rank_res_mix = rank_res_mix.sort_values(by=["aupr"],ascending=False).reset_index()
rank_res_mix.head()

,index,name,aupr,metrics
0,20,2024-08-31-06:51:43,0.3754,transformer_lr=0.0005;predictor_lr=0.0005;n=10...
1,25,2024-08-31-09:48:48,0.3719,transformer_lr=0.0005;predictor_lr=0.001;n=20;...
2,26,2024-08-31-10:45:11,0.3718,transformer_lr=0.0005;predictor_lr=0.0005;n=20...
3,19,2024-08-31-06:19:30,0.3710,transformer_lr=0.0005;predictor_lr=0.001;n=10;...
4,28,2024-08-31-12:37:22,0.3703,transformer_lr=0.0001;predictor_lr=0.0005;n=20...


write to config file

In [1]:
import yaml

In [3]:
def write_indep_test_config(mix_model_path, cancer_specific_model_path):

    indep_test_config = {
        'EXPERIMENT_DIR': './experiment',
        'SAVED_DATA_DIR': './data/saved_data',
        'sc_dir': '/data/xinliu/GeneSentence/TISCH2/',
        'SL_data_dir': './data/SL_data',

        'task': {
            'type': 'independent_test',
            'model': {
                'mix': {
                    'model_type': 'transformer',
                    'path': mix_model_path
                },
                'LAML': {
                    'model_type': 'transformer',
                    'path': cancer_specific_model_path
                },
                'LUAD': {
                    'model_type': 'transformer',
                    'path': cancer_specific_model_path
                },
                'COAD': {
                    'model_type': 'transformer',
                    'path': cancer_specific_model_path
                },
            }
        },

        'SL_dataset': {
            'general': {'path': './data/SL_data/raw/SLlabel_general.xlsx'},
            'DBKO_study2': {
                'path': './data/SL_data/processed/SLlabel_DBKO_study2.csv',
                'context': 'LAML',
                'label_type': 'rank',
            },
            'primpartner_study2_viability': {
                'path': './data/SL_data/processed/SLlabel_primpartner_study2_viability.csv',
                'context': 'LUAD',
                'label_type': 'rank',
            },
            'primpartner_study3_COAD': {
                'path': './data/SL_data/processed/SLlabel_primpartner_study3_COAD.csv',
                'context': 'COAD',
                'label_type': 'rank',
            },
            'paralog_study1_score': {
                'path': './data/SL_data/processed/SLlabel_paralog_study1_score.csv',
                'context': 'mix',
                'label_type': 'rank',
            },
            'paralog_study2_score': {
                'path': './data/SL_data/processed/SLlabel_paralog_study2_score.csv',
                'context': 'mix',
                'label_type': 'rank',
            } 
        }

    }

    return indep_test_config
    

In [18]:
indep_test_cfg_dir = "./config/indep_test_cfg"
if not os.path.exists(indep_test_cfg_dir):
    os.makedirs(indep_test_cfg_dir)

for i1, path1 in enumerate(rank_res_cancer_specific['name']):
    full_path1 = './experiment/cancer_specific/transformer/'+path1
    exp_path2 = './experiment/mix/transformer/'+rank_res_mix['name'][0]
    cfg = write_indep_test_config(mix_model_path=exp_path2, cancer_specific_model_path=full_path1)
    with open(os.path.join(indep_test_cfg_dir, f'independent_test_cancer_specific_{i1}.yaml'), 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False, indent=2, width=120)

for i2, path2 in enumerate(rank_res_mix['name']):
    full_path2 = './experiment/mix/transformer/'+path2
    exp_path1 = './experiment/cancer_specific/transformer/'+rank_res_cancer_specific['name'][0]
    cfg = write_indep_test_config(mix_model_path=full_path2, cancer_specific_model_path=exp_path1)
    with open(os.path.join(indep_test_cfg_dir, f'independent_test_mix_{i2}.yaml'), 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False, indent=2, width=120)

* 运行sh ./config/indep_test_run_all_cfg.sh会遍历indep_test_cfg_dir下所有的config文件
* 输出的perform_diff表示slformer和geneformer的所有ndcg score结果之差，可以用来估计最后画图结果
* independent_test的结果保存在'./experiment/independent_test'，每次用不同的config文件会覆盖之前的结果，画图前用选择好的checkpoint跑一次结果后再画图

#### independent test plot

In [ ]:
import os
import pandas as pd

In [ ]:
eval_cv_dir = "experiment/independent_test/eval_cv"
task_dir = {
    "cancer_specific": ["COAD", "LAML", "LUAD"],
    "mix": ["mix"]
}
models = ["geneformer", "transformer"]
metrics = ["ndcg@10","ndcg@20","ndcg@30","ndcg@50","ndcg@100"]

In [ ]:
plot_data = []

for task, dirs in task_dir.items():
    for dir in dirs:
        for f in os.listdir(os.path.join(eval_cv_dir, dir)):
            for model in models:
                if model in f and "binary" not in f:
                    df = pd.read_csv(os.path.join(eval_cv_dir, dir, f))
                    
                    for i in range(len(df)):
                        plot_data_dict = {
                            "task": task,
                            "model": model if model=="geneformer" else "SLformer"
                        }
                        for m in metrics:
                            plot_data_dict[m] = df.iloc[i][m]
                        plot_data.append(plot_data_dict)

plot_data = pd.DataFrame(plot_data)
plot_data.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [ ]:
# TASK = "cancer_specific"
TASK = "mix"

plt.figure(dpi=150)
df_list_new = []
for metric in ["ndcg@10","ndcg@20","ndcg@30","ndcg@50","ndcg@100"]:
    df_1m = plot_data[['task', 'model', metric]].copy()
    df_1m.columns = ['task', 'model', 'ndcg_score']
    df_1m['metric'] = metric
    df_list_new.append(df_1m)
plot_data_sep = pd.concat(df_list_new, ignore_index=True)
plot_data_sep_task = plot_data_sep[plot_data_sep['task']==TASK]

sns.boxplot(x='metric',y='ndcg_score',hue='model',data=plot_data_sep_task,
            palette='Set2',saturation=0.8)
if TASK == "cancer_specific":
    plt.xlabel('cancer specific independent test')
elif TASK == "mix":
    plt.xlabel('mixed cancer independent test')

plt.ylabel('ndcg score')

## add p-value
# 计算每个 metric 下两个 model 之间的 t-test p-value
metrics = plot_data_sep_task['metric'].unique()
for metric in metrics:
    data_metric = plot_data_sep_task[plot_data_sep_task['metric'] == metric]
    
    model_types = data_metric['model'].unique()
    model_1 = data_metric[data_metric['model'] == 'SLformer']['ndcg_score']
    model_2 = data_metric[data_metric['model'] == 'geneformer']['ndcg_score']
    
    # 计算 t-test p-value
    t_stat, p_value = stats.wilcoxon(model_1, model_2)
    
    # 在 boxplot 上方显示 p-value
    x1, x2 = np.where(metrics == metric)[0][0] - 0.2, np.where(metrics == metric)[0][0] + 0.2  # 调整x轴位置
    y, h, col = max(data_metric['ndcg_score']) + 0.05, 0.05, 'k'
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    plt.text((x1 + x2) * .5, y + h, f"p = {p_value:.3e}", ha='center', va='bottom', color=col)


plt.show()

#### Cross-cancer hyperparams

In [19]:
import os
import json
import pandas as pd
import numpy as np

In [20]:
# result_dir = "experiment/cross_cancer_all/transformer"
result_dir = "experiment/cross_cancer/transformer"

In [24]:
record = [d.name for d in os.scandir(result_dir) if d.is_dir()]
record = [r for r in record if r.startswith("2024-08-30") or r.startswith("2024-08-31")]
record = sorted(record, key=str.lower)[:-1]
print(record)
rank_res = []
for r in record:
    result = pd.read_csv(os.path.join(result_dir, r, "result/train_result_cross_cancer.csv"))
    avg_aupr = float(result["test_aupr"].iloc[-1].split(' ')[0])
    with open(os.path.join(result_dir, r, "params.json"), 'r') as f:
        params = json.load(f)
    rank_res.append({
        'name': r,
        'aupr': avg_aupr,
        'metrics': "transformer_lr="+str(params['transformer_lr'])+";predictor_lr="+str(params['predictor_lr'])+";n="+str(params['n'])+";num_layers="+str(params['num_layers']),
    })

['2024-08-30-10:24:21', '2024-08-30-11:20:24', '2024-08-30-12:10:58', '2024-08-30-13:05:00', '2024-08-30-14:00:11', '2024-08-30-14:52:35', '2024-08-30-15:46:22', '2024-08-30-17:08:59', '2024-08-30-18:31:53', '2024-08-30-19:41:33', '2024-08-30-21:17:15', '2024-08-30-22:44:47', '2024-08-31-00:05:36', '2024-08-31-03:54:44', '2024-08-31-09:31:29', '2024-08-31-10:38:23', '2024-08-31-11:49:08', '2024-08-31-12:52:00', '2024-08-31-13:44:54', '2024-08-31-14:48:28', '2024-08-31-15:45:43', '2024-08-31-17:19:46', '2024-08-31-18:47:49', '2024-08-31-20:15:07', '2024-08-31-21:43:20']


In [25]:
rank_res_cross_cancer = pd.DataFrame(rank_res)
rank_res_cross_cancer = rank_res_cross_cancer.sort_values(by=["aupr"],ascending=False).reset_index()
rank_res_cross_cancer.head()

,index,name,aupr,metrics
0,6,2024-08-30-15:46:22,0.2133,transformer_lr=0.0005;predictor_lr=0.001;n=20;...
1,0,2024-08-30-10:24:21,0.2095,transformer_lr=0.0005;predictor_lr=0.001;n=10;...
2,20,2024-08-31-15:45:43,0.2029,transformer_lr=0.0005;predictor_lr=0.001;n=20;...
3,15,2024-08-31-10:38:23,0.1999,transformer_lr=0.0005;predictor_lr=0.0005;n=10...
4,23,2024-08-31-20:15:07,0.1971,transformer_lr=0.0001;predictor_lr=0.0005;n=20...


In [29]:
def write_IDH1_inference_config(ckp_list):

    ckp_data = {}
    for i, path in enumerate(ckp_list):
        ckp_data[f"model_no_{i}"] = {
            'model_type': 'transformer',
            'path': os.path.join('./experiment/cross_cancer/transformer', path)
        }

    inference_config = {
        'EXPERIMENT_DIR': './experiment',
        'SAVED_DATA_DIR': './data/saved_data',
        'sc_dir': '/data/xinliu/GeneSentence/TISCH2/',
        'SL_data_dir': './data/SL_data',

        'task': {
            'type': 'inference',
            'name': 'IDH1_DDR_Glioma_test',

            'data': {
                'kegg': './data/saved_data/inference/IDH1_DDR_Glioma_kegg.npy',
                'reactome': './data/saved_data/inference/IDH1_DDR_Glioma_reactome.npy'
            },
            
            'model': ckp_data,
        },
        

        'SL_dataset': {
            'general': {'path': './data/SL_data/raw/SLlabel_general.xlsx'}
        }

    }

    return inference_config

In [30]:
ckp_list = rank_res_cross_cancer['name']
cfg = write_IDH1_inference_config(ckp_list=ckp_list)
with open('./config/IDH1_inference_test.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False, indent=2, width=120)

* 运行python slformer/main.py --config_file=./config/IDH1_inference_test.yaml --wandb_track=0，结果保存在'./experiment/inference'
* 主要关注PRKDC, NUDT1 (湿实验验证目标), PARP1, BCL2L1 (有文献支持和IDH1的SL关系)的排名